In [ ]:
#Excel Extract
 
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
import pandas as pd
from openpyxl import Workbook
import logging
from concurrent.futures import ThreadPoolExecutor, as_completed
 
# Set up logging to display in Jupyter Notebook
logger = logging.getLogger()
logger.setLevel(logging.INFO)
 
if not logger.hasHandlers():
    handler = logging.StreamHandler()
    formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    handler.setFormatter(formatter)
    logger.addHandler(handler)
 
# Chrome options
chrome_options = Options()
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("--window-size=1920x1080")
prefs = {
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True
}
chrome_options.add_experimental_option("prefs", prefs)
 

link_file = r"ICICISHEET.csv"
links_df = pd.read_csv(link_file)
 
#output.xlsx
output_file = 'output.xlsx'
wb = Workbook()
ws = wb.active
ws.title = "Scheme Data"
ws.append(["Link", "AUM", "SFIN", "Table Data"])
 
#find under DOM
def find_button_by_ids(driver, button_ids):
    for button_id in button_ids:
        try:
            button = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.ID, button_id))
            )
            if button.is_displayed():
                return button
        except TimeoutException:
            continue
    return None
 
def find_button_by_css_selector(driver):
    buttons = driver.find_elements(By.CSS_SELECTOR, '[id^="showMore"]')
    for button in buttons:
        if button.is_displayed():
            return button
    return None
 
# Function to find the button using XPath
def find_button_by_xpath(driver):
    buttons = driver.find_elements(By.XPATH, '//button[contains(@id, "showMore")]')
    for button in buttons:
        if button.is_displayed():
            return button
    return None
 
# Function to extract data from a single link
def extract_data_from_link(link, index):
    driver = webdriver.Chrome(options=chrome_options)
    driver.get(link)
    print(f"Processing link {index + 1}: {link}")
    logger.info(f"Processing link {index + 1}: {link}")
   
    try:
        # Attempt to find and click the 'Show Details' button
        button_ids = ['showMore', 'showMore1', 'showMore2', 'showMore3', 'showMore4',
                      'showMore5', 'showMore6', 'showMore7']
        show_more_button = find_button_by_ids(driver, button_ids)
        if not show_more_button:
            show_more_button = find_button_by_css_selector(driver)
        if not show_more_button:
            show_more_button = find_button_by_xpath(driver)
       
        if show_more_button:
            # Scroll to the element and click it using JavaScript
            driver.execute_script("arguments[0].scrollIntoView(true);", show_more_button)
            driver.execute_script("arguments[0].click();", show_more_button)
            logger.info(f"'Show Details' button clicked for link: {link}")
        else:
            raise Exception("Show Details button not found or not clickable.")
       
    except Exception as e:
        print(f"Error clicking 'Show Details' for link {index + 1}: {link} - {e}")
        logger.error(f"Error clicking 'Show Details' for link {index + 1}: {link} - {e}")
        driver.quit()
        return [[link, None, None, None]]  # Return as a list of list for consistency
 
    try:
        # Extract AUM value
        aum_element = driver.find_element(By.XPATH, "//td[contains(text(), 'Million')]")
        aum_value = aum_element.text.strip()
 
        # Extract SFIN value
        sfin_element = driver.find_element(By.XPATH, "//h1[contains(@class, 'gcolor')]/span[@class='font-s']")
        sfin_value = sfin_element.text.strip()
 
        # Wait for any table to be present
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.TAG_NAME, "table"))
        )
        tables = driver.find_elements(By.TAG_NAME, "table")
       
        all_table_data = []
        for table in tables:
            rows = table.find_elements(By.TAG_NAME, "tr")
            for row in rows:
                cols = row.find_elements(By.TAG_NAME, "td")
                row_data = [link, aum_value, sfin_value] + [col.text.strip() for col in cols]
                all_table_data.append(row_data)
            logger.info(f"Data saved for table in link: {link}")
       
        driver.quit()
        return all_table_data
       
    except Exception as e:
        print(f"Error extracting data from link {index + 1}: {link} - {e}")
        logger.error(f"Error extracting data from link {index + 1}: {link} - {e}")
        driver.quit()
        return [[link, None, None, None]]  # Return as a list of list for consistency
 
# Run the data extraction in parallel using ThreadPoolExecutor
with ThreadPoolExecutor(max_workers=4) as executor:  # Adjust max_workers based on your system's capability
    futures = [executor.submit(extract_data_from_link, row['Link'], index) for index, row in links_df.iterrows()]
   
    for future in as_completed(futures):
        result = future.result()
        if result is not None:
            for row_data in result:
                ws.append(row_data)
 
# Save the output Excel file
wb.save(output_file)
 
print("Data extraction completed successfully.")
logger.info("Data extraction completed successfully.")

In [ ]:
#PDF Download Script
 
import time, base64, os, re
import pandas as pd
from selenium import webdriver                                # Controls the browser
from selenium.webdriver.common.by import By                   # Locators (ID, CLASS_NAME, XPATH, etc.)
from selenium.webdriver.support.ui import WebDriverWait       # Waits for elements to appear
from selenium.webdriver.support import expected_conditions as EC  # Conditions like "visible", "clickable"
from selenium.webdriver.chrome.options import Options
from urllib.parse import urljoin
 
SELECTORS = {
    "filter_list": (By.ID, "filterlist"),
    "last_filter_li": (By.XPATH, "./li[last()]"),
    "fund_links": (By.CSS_SELECTOR, "#fund-perform-table a"),
    "portfolio_button": (By.CLASS_NAME, "portfolio"),
    "show_more": (By.ID, "showMore1"),
    "companies_table": (By.ID, "companies-table"),
    "page_header": (By.TAG_NAME, "h1"),
    "tab_content": (By.CLASS_NAME, "tab-content")
}
 
BASE_URL = r"https://www.iciciprulife.com/fund-performance/all-products-fund-performance-details.html"
PDF_FOLDER = "INSR_PDF"
XLS_FILE = "INSURANCE_MASTER_XLS.xlsx"
BATCH_SIZE = 20
COOLDOWN = 30
 
os.makedirs(PDF_FOLDER, exist_ok=True)
 
 
# --- Utility Functions ---
def clean_text(text):
    text = text.strip()
    text = re.sub(r'[\\/*?:"<>|\/]', "_", text.lower())
    return text
 
def save_pdf(driver,filename):
    pdf = driver.execute_cdp_cmd("Page.printToPDF", {
        "printBackground": True,
        "paperWidth": 8.27,
        "paperHeight": 11.69
    })
   
    save_path = os.path.join(PDF_FOLDER, f"{filename}.pdf")
    with open(save_path, "wb") as f:
        f.write(base64.b64decode(pdf['data']))
 
def save_table_to_excel(driver, content, writer):
    tables = driver.find_elements(By.TAG_NAME, "table")
    all_data = []
 
    for table in tables:
        rows = table.find_elements(By.TAG_NAME, "tr")
        for row in rows:
            cells = row.find_elements(By.XPATH, ".//th | .//td")
            row_data = [cell.text.strip() for cell in cells]
            if any(row_data):
                all_data.append(row_data)
        all_data.append([])
 
    if all_data:
        df = pd.DataFrame(all_data)
        df.insert(0,"url",content["url"])
        df.insert(1,"ins",content["name"])
        df.insert(2,"sfin",content["sfin"])
        df.to_excel(writer, sheet_name=content["name"][:31], index=False, header=False)
         
    # file_path = os.path.join(CONFIG["xls_folder"], filename)
    # df.to_excel(file_path, index=False, header=False)
 
def execute_scripts(driver, element=None, type=""):
    if type == "":
        print("Type not Correct")
        return driver
 
    REQ_SCRIPTS = {
        "color": """
            const style = document.createElement('style');
            style.type = 'text/css';
            style.innerHTML = `
                html, body {
                    background: white !important;
                    color: #111 !important;
                    -webkit-print-color-adjust: exact !important;
                    print-color-adjust: exact !important;
                }
                * {
                    color: #111 !important;
                    background: white !important;
                    border-color: #111 !important;
                }
                a, span, div, td, th, p, h1, h2, h3, h4, h5, h6 {
                    color: #111 !important;
                }
            `;
            document.head.appendChild(style);
        """
    }
 
    driverReturn = driver.execute_script(REQ_SCRIPTS[type], element) if element else driver.execute_script(REQ_SCRIPTS[type])
    time.sleep(1)
    return driverReturn
 
# --- Browser Setup ---
options = Options()
options.add_argument("start-maximized")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36")
options.add_argument("--headless")
driver = webdriver.Chrome(options=options)
driver.get(BASE_URL)
 
excelWriter = pd.ExcelWriter(XLS_FILE,engine="xlsxwriter")
 
WebDriverWait(driver, 5).until(EC.presence_of_element_located(SELECTORS["tab_content"]))
 
filter_list = driver.find_element(*SELECTORS["filter_list"])
last_all = filter_list.find_element(*SELECTORS["last_filter_li"])
last_all.click()
time.sleep(2)
 
pdf_links = driver.find_elements(*SELECTORS["fund_links"])
 
total_links = len(pdf_links)
print("Total links:", total_links)
 
for batch_start in range(0, total_links, BATCH_SIZE):
    batch = pdf_links[batch_start:batch_start + BATCH_SIZE]
    print(f"\nProcessing batch {(batch_start // BATCH_SIZE) + 1} with {len(batch)} links")
 
    for j, link in enumerate(batch):
        i = batch_start + j
        try:
            driver.execute_script("window.open(arguments[0].href, '_blank');", link)
            WebDriverWait(driver, 5).until(lambda d: len(d.window_handles) == 2)
            driver.switch_to.window(driver.window_handles[-1])
           
            try:
                performance_element = WebDriverWait(driver, 8).until(
                    EC.element_to_be_clickable(SELECTORS["portfolio_button"])
                )
                performance_element.click()
                WebDriverWait(driver, 8).until(lambda d: len(d.window_handles) == 3)
                driver.switch_to.window(driver.window_handles[-1])
                time.sleep(2)
            except:
               
                print("Portfolio click failed or not present")
           
            try:driver.find_element(*SELECTORS["show_more"]).click()
            except:driver.execute_script("document.getElementById('showMore1').click();")
 
            WebDriverWait(driver, 8).until(
                EC.presence_of_element_located(SELECTORS["companies_table"])
            )
 
            execute_scripts(driver, type="color")
       
            header = driver.find_element(*SELECTORS["page_header"]).text
            file_name, sfin = header.split("\n")
            content = {
                "url":driver.current_url,
                "name": clean_text(file_name),
                "sfin": sfin
            }
           
            print(f"[{i+1}] INSR NAME: {content["name"]} PDF URL: {content["url"]}")
 
            save_pdf(driver,file_name)
            save_table_to_excel(driver,content,excelWriter)
 
        except Exception as e:
            print(f"[FAIL] {file_name}")
            print(f"[{i+1}] Failed: {e}")
        finally:
            while len(driver.window_handles) > 1:
               
                driver.switch_to.window(driver.window_handles[-1])
                driver.close()
            driver.switch_to.window(driver.window_handles[0])
 
    if batch_start + BATCH_SIZE < total_links:
        print(f"Sleeping {COOLDOWN} seconds before next batch...")
        time.sleep(COOLDOWN)
       
excelWriter.close()

In [ ]:
import time, base64, os, re
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options


class ICICIInsuranceScraper:

    BASE_URL = "https://www.iciciprulife.com/fund-performance/all-products-fund-performance-details.html"
    PDF_FOLDER = "INSR_PDF"
    XLS_FILE = "INSURANCE_MASTER_XLS.xlsx"
    BATCH_SIZE = 20
    COOLDOWN = 30

    SELECTORS = {
        "filter_list": (By.ID, "filterlist"),
        "last_filter_li": (By.XPATH, "./li[last()]"),
        "fund_links": (By.CSS_SELECTOR, "#fund-perform-table a"),
        "portfolio_button": (By.CLASS_NAME, "portfolio"),
        "show_more": (By.ID, "showMore1"),
        "companies_table": (By.ID, "companies-table"),
        "page_header": (By.TAG_NAME, "h1"),
        "tab_content": (By.CLASS_NAME, "tab-content")
    }

    # ---------------- INIT ----------------
    def __init__(self):

        os.makedirs(self.PDF_FOLDER, exist_ok=True)

        options = Options()
        options.add_argument("start-maximized")
        options.add_argument("--headless")
        options.add_argument(
            "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
        )

        self.driver = webdriver.Chrome(options=options)
        self.wait = WebDriverWait(self.driver, 10)
        self.writer = pd.ExcelWriter(self.XLS_FILE, engine="xlsxwriter")

    # ---------------- CLEAN ----------------
    def clean_text(self, text):
        text = text.strip()
        return re.sub(r'[\\/*?:"<>|]', "_", text.lower())

    # ---------------- PDF SAVE ----------------
    def save_pdf(self, filename):

        pdf = self.driver.execute_cdp_cmd(
            "Page.printToPDF",
            {
                "printBackground": True,
                "paperWidth": 8.27,
                "paperHeight": 11.69
            }
        )

        path = os.path.join(self.PDF_FOLDER, f"{filename}.pdf")
        with open(path, "wb") as f:
            f.write(base64.b64decode(pdf['data']))

    # ---------------- TABLE SAVE ----------------
    def save_table_to_excel(self, content):

        tables = self.driver.find_elements(By.TAG_NAME, "table")
        all_data = []

        for table in tables:
            rows = table.find_elements(By.TAG_NAME, "tr")

            for row in rows:
                cells = row.find_elements(By.XPATH, ".//th | .//td")
                row_data = [c.text.strip() for c in cells]
                if any(row_data):
                    all_data.append(row_data)

            all_data.append([])

        if all_data:
            df = pd.DataFrame(all_data)
            df.insert(0, "url", content["url"])
            df.insert(1, "ins", content["name"])
            df.insert(2, "sfin", content["sfin"])

            df.to_excel(
                self.writer,
                sheet_name=content["name"][:31],
                index=False,
                header=False
            )

    # ---------------- PRINT COLOR FIX ----------------
    def apply_print_css(self):

        script = """
            const style = document.createElement('style');
            style.innerHTML = `
                html, body {background: white !important; color: #111 !important;}
                * {color: #111 !important; background: white !important;}
            `;
            document.head.appendChild(style);
        """

        self.driver.execute_script(script)
        time.sleep(1)

    # ---------------- PROCESS ONE LINK ----------------
    def process_link(self, link, index):

        file_name = "UNKNOWN"

        try:
            self.driver.execute_script(
                "window.open(arguments[0].href, '_blank');", link
            )

            self.wait.until(lambda d: len(d.window_handles) == 2)
            self.driver.switch_to.window(self.driver.window_handles[-1])

            try:
                perf = self.wait.until(
                    EC.element_to_be_clickable(self.SELECTORS["portfolio_button"])
                )
                perf.click()

                self.wait.until(lambda d: len(d.window_handles) == 3)
                self.driver.switch_to.window(self.driver.window_handles[-1])
                time.sleep(2)

            except:
                print("Portfolio button missing")

            try:
                self.driver.find_element(*self.SELECTORS["show_more"]).click()
            except:
                self.driver.execute_script(
                    "document.getElementById('showMore1').click();"
                )

            self.wait.until(
                EC.presence_of_element_located(self.SELECTORS["companies_table"])
            )

            self.apply_print_css()

            header = self.driver.find_element(
                *self.SELECTORS["page_header"]
            ).text

            file_name, sfin = header.split("\n")

            content = {
                "url": self.driver.current_url,
                "name": self.clean_text(file_name),
                "sfin": sfin
            }

            print(f"[{index+1}] {content['name']}")

            self.save_pdf(file_name)
            self.save_table_to_excel(content)

        except Exception as e:
            print(f"[FAIL] {file_name} -> {e}")

        finally:
            while len(self.driver.window_handles) > 1:
                self.driver.switch_to.window(self.driver.window_handles[-1])
                self.driver.close()
            self.driver.switch_to.window(self.driver.window_handles[0])

    # ---------------- MAIN RUNNER ----------------
    def run(self):

        self.driver.get(self.BASE_URL)

        self.wait.until(
            EC.presence_of_element_located(self.SELECTORS["tab_content"])
        )

        filter_list = self.driver.find_element(
            *self.SELECTORS["filter_list"]
        )

        last_all = filter_list.find_element(
            *self.SELECTORS["last_filter_li"]
        )

        last_all.click()
        time.sleep(2)

        links = self.driver.find_elements(*self.SELECTORS["fund_links"])
        total = len(links)

        print("TOTAL LINKS:", total)

        for start in range(0, total, self.BATCH_SIZE):

            batch = links[start:start + self.BATCH_SIZE]
            print(f"\nBATCH {(start // self.BATCH_SIZE) + 1}")

            for j, link in enumerate(batch):
                self.process_link(link, start + j)

            if start + self.BATCH_SIZE < total:
                print(f"Sleeping {self.COOLDOWN} sec...")
                time.sleep(self.COOLDOWN)

        self.writer.close()
        self.driver.quit()
        print("✅ DONE")


# ---------------- RUN ----------------
if __name__ == "__main__":
    scraper = ICICIInsuranceScraper()
    scraper.run()